# Data Preparation

This notebook turns the raw EMSCAD job postings dataset into a single cleaned dataset. The text fields are combined and cleaned, categorical fields with meaningful missing data are filled with a fixed placeholder, and columns that carry no usable signal are dropped. The result is saved to `data/processed/cleaned.csv`. No fitting, splitting, or modelling is done in this notebook. Those steps are taken in `03_modelling_classical.ipynb`.

In [7]:
# Standard library
import sys

# Third-party
import pandas as pd

# Make the repository root importable so src/ can be reached from notebooks/
sys.path.append("..")

# Local
from src.clean_text import TEXT_FIELDS, clean_text, combine_fields

## Load the Raw Dataset

The same raw CSV file used in `01_data_understanding.ipynb` is loaded and its shape is checked to confirm it still matches the 17,880 rows and 18 columns confirmed in `01_data_understanding.ipynb` before any preparation begins.

In [8]:
# Load the job postings dataset and report its shape
df = pd.read_csv("../data/raw/fake_job_postings.csv")
print(df.shape)

(17880, 18)


## Combine and Clean the Text Fields

`01_data_understanding.ipynb` found that the text fields (`title`, `company_profile`, `description`, `requirements`, `benefits`) contain words fused together where HTML tags or line breaks were stripped without leaving a space, as well as raw HTML entities such as `&amp;`. `combine_fields` joins these fields into one `text` column per posting, treating a missing field as empty. `clean_text` lower-cases it, unescapes HTML entities, removes HTML tags, replaces links with the placeholder `url`, and removes remaining punctuation. Both functions are stored in `src/clean_text.py` to ensure that the same cleaning logic can later be imported by the app.

In [9]:
# Combine the text fields into one column per posting, then clean it
df["text"] = df.apply(combine_fields, axis=1)
df["text"] = df["text"].apply(clean_text)

The fraudulent posting example from `01_data_understanding.ipynb`, which contained the raw HTML entity `&amp;`, is used to check that the cleaning step behaves as expected on real data. The same combined text (`title`, `company_profile`, `description`, `requirements`, `benefits`, in that order) is printed before and after cleaning so both sides show the same underlying content and only the effect of `clean_text` is visible.

In [10]:
# Compare the raw combined text and the cleaned combined text for the same fraudulent posting inspected in notebook 01
fraudulent_row = df.loc[df["fraudulent"] == 1].iloc[0]
raw_combined_text = combine_fields(fraudulent_row)
print("Raw combined text:\n", raw_combined_text[:300])
print("\nCleaned combined text:\n", fraudulent_row["text"][:300])

Raw combined text:
 IC&E Technician                                                                                  Staffing &amp; Recruiting done right for the Oil &amp; Energy Industry!Represented candidates are automatically granted the following perks: Expert negotiations on your behalf, maximizing your compensati

Cleaned combined text:
 ic e technician staffing recruiting done right for the oil energy industry represented candidates are automatically granted the following perks expert negotiations on your behalf maximizing your compensation package and implimenting ongoing increases significant signing bonus by refined resources in


## Fill Missing Categorical Values

`01_data_understanding.ipynb` identified ten columns with meaningful missing data. Of these, `salary_range` is handled separately (see the "Assemble and Save the Cleaned Dataset" section below) because it also has too many unique values to work as a normal category. The remaining nine: `department`, `required_education`, `benefits`, `required_experience`, `function`, `industry`, `employment_type`, `company_profile`, and `requirements` are filled with the fixed placeholder string `"missing"`. This turns a missing value into its own category instead of dropping the row or guessing a value. Whether a poster left the field blank is information the model can learn from. Fraudulent postings may skip different fields compared to legitimate ones. Dropping every row with a missing value will discard most of the dataset since missing values are spread across many columns. Standard imputation (filling in the mean or most common value) will erase this signal by making every missing entry look the same as the most typical filled-in one. The count of missing values per column is printed before and after the fill to confirm none remain.

In [11]:
# Fill missing categorical values with a fixed placeholder string
PLACEHOLDER = "missing"
columns_with_missing_values = ["department", "required_education", "benefits", "required_experience", "function", "industry", "employment_type", "company_profile", "requirements"]

print("Missing values before fill:")
print(df[columns_with_missing_values].isna().sum())

df[columns_with_missing_values] = df[columns_with_missing_values].fillna(PLACEHOLDER)

print("\nMissing values after fill:")
print(df[columns_with_missing_values].isna().sum())

Missing values before fill:
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
dtype: int64

Missing values after fill:
department             0
required_education     0
benefits               0
required_experience    0
function               0
industry               0
employment_type        0
company_profile        0
requirements           0
dtype: int64


## Assemble and Save the Cleaned Dataset

The cleaned dataset keeps the label (`fraudulent`), the cleaned `text` column, the metadata fields (now free of missing values), and the binary flag columns (`telecommuting`, `has_company_logo`, `has_questions`). Three columns are dropped: `job_id` is a row identifier with no modelling signal, `location` combines country, state, and city into one free-text string (for example `"US, CA, San Francisco"`) with 3,105 distinct values, and `salary_range` is missing in 83.96% of postings and has 874 distinct values. `location` and `salary_range` both behave more like free text or identifiers instead of categories. Filling `salary_range` with a placeholder will leave one `"missing"` category representing most postings and hundreds of near-unique values for the rest, which is not usable as a categorical feature. The result is saved to `data/processed/cleaned.csv`, the single input to the modelling notebooks.

In [12]:
# Assemble the final set of columns for the cleaned dataset
metadata_fields = ["department", "employment_type", "required_experience", "required_education", "industry", "function"]
binary_fields = ["telecommuting", "has_company_logo", "has_questions"]
cleaned_df = df[["fraudulent", "text"] + metadata_fields + binary_fields]
print(cleaned_df.shape)

# Save the cleaned dataset
cleaned_df.to_csv("../data/processed/cleaned.csv", index=False)

(17880, 11)


## Summary

- Combined the five text fields (`title`, `company_profile`, `description`, `requirements`, `benefits`) into one `text` column and cleaned it using the shared `clean_text` function, resolving the fused-word and HTML-entity issues found in `01_data_understanding.ipynb`.
- Filled nine columns with meaningful missing data using the fixed placeholder `"missing"`, treating missing values as a category instead of dropping rows or imputing a guessed value, and confirmed that there are no more missing values in the DataFrame.
- Dropped `job_id` (no modelling signal), `location` (3,105 distinct values combining country, state, and city into one free-text string), and `salary_range` (83.96% missing and 874 distinct values, behaving like free text instead of a category) before saving.
- Saved the cleaned dataset to `data/processed/cleaned.csv`, containing the label, the cleaned text, the remaining metadata fields, and the binary flags.
- No column was fitted, transformed with a statistic learned from data, or split. `03_modelling_classical.ipynb` performs the train/validation/test split.